In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt
/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4


In [13]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [14]:
!git clone https://github.com/abewley/sort.git

fatal: destination path 'sort' already exists and is not an empty directory.


In [15]:
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import math

# ---- LOAD SORT ----
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ---- MODEL ----
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt")

# ---- VIDEO ----
cap = cv2.VideoCapture("/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4")

FPS = 100
dt = 1 / FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_final_with_speed.mp4", fourcc, FPS, (960,540))

tracker = Sort()

# ---- STORAGE ----
track_history = defaultdict(lambda: deque(maxlen=10))
track_speed = defaultdict(float)
track_prev_speed = defaultdict(float)
track_acc = defaultdict(float)

csv_data = []

# ---- HOMOGRAPHY ----
scale_x = 960 / 1920
scale_y = 540 / 1080

img_pts = np.array([
    [540*scale_x,1066*scale_y],
    [1650*scale_x,1066*scale_y],
    [972*scale_x,216*scale_y],
    [1080*scale_x,216*scale_y]
], dtype=np.float32)

depth = 74.65
width = 3.0

world_pts = np.array([
    [0,0],[width,0],[0,depth],[width,depth]
], dtype=np.float32)

H, _ = cv2.findHomography(img_pts, world_pts)

# ---- FUNCTIONS ----

def pixel_to_world(u, v):
    pt = np.array([u, v, 1]).reshape(3,1)
    world = H @ pt
    world /= world[2]
    return world[0][0], world[1][0]

def get_bottom_center(box):
    x1, y1, x2, y2 = box
    return (x1 + x2) / 2, y2

def compute_speed(track_id, Y):
    hist = track_history[track_id]
    hist.append(Y)

    if len(hist) < 5:
        return 0

    dy = hist[-1] - hist[0]
    dt_total = (len(hist)-1) * dt

    speed = abs(dy) / dt_total * 3.6
    speed = 0.7*track_speed[track_id] + 0.3*speed
    track_speed[track_id] = speed

    return speed

def get_mttc_risk(mttc):
    if mttc < 1.5:
        return "HIGH", (0,0,255)
    elif mttc < 3:
        return "MED", (0,255,255)
    else:
        return "LOW", (0,255,0)

def get_drac_risk(drac):
    if drac > 3.4:
        return "HIGH", (0,0,255)
    elif drac > 2:
        return "MED", (0,255,255)
    else:
        return "LOW", (0,255,0)

# ---- MAIN ----
frame_count = 0
MAX_SECONDS = 30

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count >= FPS * MAX_SECONDS:
        break

    frame = cv2.resize(frame, (960,540))

    # ---- ROI ----
    roi_pts = np.array([
        [540*scale_x,1066*scale_y],
        [1650*scale_x,1066*scale_y],
        [1080*scale_x,216*scale_y],
        [972*scale_x,216*scale_y]
    ], dtype=np.int32)

    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [roi_pts], 255)
    frame_masked = cv2.bitwise_and(frame, frame, mask=mask)

    cv2.polylines(frame, [roi_pts], True, (255,0,0), 2)

    track_positions = {}

    # ---- DETECTION ----
    results = model(frame_masked, imgsz=640, conf=0.4, verbose=False)[0]

    detections = []
    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls not in [0,2] or conf < 0.5:
            continue

        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
        detections.append([x1,y1,x2,y2,conf])

    dets = np.array(detections) if detections else np.empty((0,5))
    tracks = tracker.update(dets)

    # ---- TRACK PROCESSING ----
    for trk in tracks:
        x1,y1,x2,y2,track_id = trk
        track_id = int(track_id)

        u,v = get_bottom_center((x1,y1,x2,y2))
        if v < 120:
            continue

        X,Y = pixel_to_world(u,v)

        speed = compute_speed(track_id, Y)

        prev_speed = track_prev_speed[track_id]
        acc = ((speed - prev_speed)/3.6) / dt
        acc = 0.7*track_acc[track_id] + 0.3*acc

        track_prev_speed[track_id] = speed
        track_acc[track_id] = acc

        track_positions[track_id] = {
            "X":X,"Y":Y,"speed":speed,"acc":acc
        }

        # 🔥 DRAW SPEED
        cv2.rectangle(frame,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
        cv2.putText(frame,
                    f"{speed:.1f} km/h",
                    (int(x1), int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,(0,255,255),2)

    # ---- MTTC + DRAC ----
    vehicles = sorted(track_positions.items(), key=lambda x: x[1]["Y"])

    for i in range(len(vehicles)-1):
        rear_id, rear = vehicles[i]
        front_id, front = vehicles[i+1]

        if abs(rear["X"] - front["X"]) > 1.5:
            continue

        Vr = rear["speed"]/3.6
        Vf = front["speed"]/3.6
        ar = rear["acc"]
        af = front["acc"]

        dv = Vr - Vf
        da = ar - af
        S = abs(front["Y"] - rear["Y"])

        if dv <= 0 or S == 0:
            continue

        # ---- MTTC ----
        if abs(da) < 1e-3:
            mttc = S/dv
        else:
            disc = dv**2 + 2*da*S
            if disc < 0:
                continue
            t1 = (dv + math.sqrt(disc))/da
            t2 = (dv - math.sqrt(disc))/da
            valid = [t for t in [t1,t2] if t>0]
            if not valid:
                continue
            mttc = min(valid)

        # ---- DRAC ----
        drac = (dv**2) / S

        mttc_level, mttc_color = get_mttc_risk(mttc)
        drac_level, drac_color = get_drac_risk(drac)

        cv2.putText(frame,
                    f"MTTC:{mttc:.2f}s {mttc_level}",
                    (50,50+30*i),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,mttc_color,2)

        cv2.putText(frame,
                    f"DRAC:{drac:.2f} {drac_level}",
                    (400,50+30*i),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,drac_color,2)

        csv_data.append({
            "frame":frame_count,
            "rear":rear_id,
            "front":front_id,
            "mttc":mttc,
            "drac":drac
        })

    out.write(frame)
    frame_count += 1

# ---- SAVE CSV ----
pd.DataFrame(csv_data).to_csv("final_metrics.csv", index=False)

cap.release()
out.release()

print("FINAL: Speed + MTTC + DRAC all working")

✅ FINAL: Speed + MTTC + DRAC all working


In [4]:
import sys
sys.path.append("./sort")

In [5]:
!pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=b4547ee17077b6319bcc0214f26b0dbcc8353b204ea0361a09152fd36320b340
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [6]:
import sys
sys.path.insert(0, "/kaggle/working/sort")

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import math

# ---- LOAD SORT ----
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ---- MODEL ----
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt")

# ---- VIDEO ----
cap = cv2.VideoCapture("/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4")

FPS = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1: FPS = 100
dt = 1/FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("finalmaybe_output.mp4", fourcc, FPS, (960,540))

tracker = Sort()

# ---- STORAGE ----
track_history_world = defaultdict(lambda: deque(maxlen=20))
track_history_px = defaultdict(lambda: deque(maxlen=20))

# ---- HOMOGRAPHY ----
scale_x = 960/1920
scale_y = 540/1080

img_pts = np.array([
    [540*scale_x,1066*scale_y],
    [1650*scale_x,1066*scale_y],
    [972*scale_x,216*scale_y],
    [1080*scale_x,216*scale_y]
], dtype=np.float32)

world_pts = np.array([
    [0,0],[3,0],[0,74.65],[3,74.65]
], dtype=np.float32)

H,_ = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

# ---- FUNCTIONS ----
def pixel_to_world(u,v):
    pt = np.array([u,v,1]).reshape(3,1)
    w = H@pt; w/=w[2]
    return w[0][0], w[1][0]

def world_to_pixel(X,Y):
    pt = np.array([X,Y,1]).reshape(3,1)
    img = H_inv@pt; img/=img[2]
    return int(img[0][0]), int(img[1][0])

def smooth_points(points):
    pts = list(points)
    sm=[]
    for i in range(len(pts)):
        xs=[p[0] for p in pts[max(0,i-3):i+1]]
        ys=[p[1] for p in pts[max(0,i-3):i+1]]
        sm.append((int(np.mean(xs)),int(np.mean(ys))))
    return sm

def compute_velocity(hist):
    vel=np.zeros(2); count=0
    for i in range(1,min(6,len(hist))):
        vel+=(hist[-i]-hist[-i-1])/dt
        count+=1
    return vel/max(count,1)

def draw_axes(frame):
    # X axis ticks (meters)
    for x in np.arange(0,3.1,0.5):
        p1 = world_to_pixel(x,0)
        p2 = world_to_pixel(x,1)
        cv2.line(frame,p1,p2,(255,0,0),1)
        if int(x)==x:
            cv2.putText(frame,f"{int(x)}m",p1,cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,0,0),1)

    # Y axis ticks
    for y in range(0,75,10):
        p1 = world_to_pixel(0,y)
        p2 = world_to_pixel(1,y)
        cv2.line(frame,p1,p2,(0,255,0),1)
        cv2.putText(frame,f"{y}m",p1,cv2.FONT_HERSHEY_SIMPLEX,0.4,(0,255,0),1)

def draw_traj(frame, pts):
    for i in range(1,len(pts)):
        if i%2==0:
            cv2.line(frame, pts[i-1], pts[i], (255,0,255),2)

def draw_glow(frame, center):
    for r in range(20,0,-4):
        overlay=frame.copy()
        cv2.circle(overlay,center,r,(0,0,255),-1)
        cv2.addWeighted(overlay,0.2,frame,0.8,0,frame)

# ---- MAIN ----
while cap.isOpened():
    ret,frame=cap.read()
    if not ret: break

    frame=cv2.resize(frame,(960,540))

    #  ROI MASK RESTORED
    roi_pts=np.array([
        [540*scale_x,1066*scale_y],
        [1650*scale_x,1066*scale_y],
        [1080*scale_x,216*scale_y],
        [972*scale_x,216*scale_y]
    ],dtype=np.int32)

    mask=np.zeros(frame.shape[:2],dtype=np.uint8)
    cv2.fillPoly(mask,[roi_pts],255)
    frame_masked=cv2.bitwise_and(frame,frame,mask=mask)

    cv2.polylines(frame,[roi_pts],True,(255,0,0),2)

    #  AXES RESTORED
    draw_axes(frame)

    results=model(frame_masked)[0]

    detections=[]
    for b in results.boxes:
        x1,y1,x2,y2=b.xyxy[0].cpu().numpy()
        detections.append([x1,y1,x2,y2,1.0])

    tracks=tracker.update(np.array(detections) if detections else np.empty((0,5)))

    vehicles={}

    for t in tracks:
        x1,y1,x2,y2,tid=t
        tid=int(tid)

        u=(x1+x2)/2; v=y2
        X,Y=pixel_to_world(u,v)

        hist=track_history_world[tid]
        hist.append(np.array([X,Y]))

        vel=compute_velocity(hist)

        vehicles[tid]={"pos":np.array([X,Y]),"vel":vel,"px":(int(u),int(v))}

        track_history_px[tid].append((int(u),int(v)))
        pts=smooth_points(track_history_px[tid])
        draw_traj(frame,pts)

    #  SINGLE NEAREST COLLISION
    for id1,v1 in vehicles.items():
        best=None; best_t=1e9

        for id2,v2 in vehicles.items():
            if id1==id2: continue

            rel_p=v2["pos"]-v1["pos"]
            rel_v=v2["vel"]-v1["vel"]

            if np.dot(rel_v,rel_v)<1e-6: continue

            t=-np.dot(rel_p,rel_v)/np.dot(rel_v,rel_v)

            if t<=0 or t>5: continue

            dist=np.linalg.norm((v2["pos"]+v2["vel"]*t)-(v1["pos"]+v1["vel"]*t))
            if dist>2: continue

            if t<best_t:
                best_t=t
                best=(id2,v2,t)

        if best is None: continue

        id2,v2,t=best
        cp=v1["pos"]+v1["vel"]*t
        px=world_to_pixel(cp[0],cp[1])

        draw_glow(frame,px)
        cv2.line(frame,v1["px"],px,(0,0,255),2)
        cv2.line(frame,v2["px"],px,(0,0,255),2)

    out.write(frame)

cap.release()
out.release()

print(" FINAL: restored + stable + correct")


0: 384x640 2 cars, 2 2wheelers, 25.6ms
Speed: 2.3ms preprocess, 25.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.3ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.6ms preprocess, 24.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 25.0ms
Speed: 2.6ms preprocess, 25.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 20.4ms
Speed: 2.2ms preprocess, 20.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 20.5ms
Speed: 1.9ms preprocess, 20.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 20.5ms
Speed: 1.9ms preprocess, 20.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 20.4ms
S

In [7]:
sort_path = "/kaggle/working/sort/sort.py"

# Read file
with open(sort_path, "r") as f:
    lines = f.readlines()

# Remove matplotlib-related lines
new_lines = []
for line in lines:
    if "matplotlib" in line or "plt." in line:
        continue
    new_lines.append(line)

# Write cleaned file
with open(sort_path, "w") as f:
    f.writelines(new_lines)

print(" sort.py cleaned (matplotlib removed)")

✅ sort.py cleaned (matplotlib removed)


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import math

# ---- LOAD SORT ----
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ---- MODEL ----
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt")

# ---- VIDEO ----
cap = cv2.VideoCapture("/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4")

FPS = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1: FPS = 100
dt = 1/FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("finalmaybeeemaybee_output.mp4", fourcc, FPS, (960,540))

tracker = Sort()

# ---- STORAGE ----
track_history_world = defaultdict(lambda: deque(maxlen=20))
track_history_px = defaultdict(lambda: deque(maxlen=20))
track_speed = defaultdict(float)

# ---- HOMOGRAPHY ----
scale_x = 960/1920
scale_y = 540/1080

img_pts = np.array([
    [540*scale_x,1066*scale_y],
    [1650*scale_x,1066*scale_y],
    [972*scale_x,216*scale_y],
    [1080*scale_x,216*scale_y]
], dtype=np.float32)

world_pts = np.array([
    [0,0],[3,0],[0,74.65],[3,74.65]
], dtype=np.float32)

H,_ = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

# ---- FUNCTIONS ----
def pixel_to_world(u,v):
    pt = np.array([u,v,1]).reshape(3,1)
    w = H@pt; w/=w[2]
    return w[0][0], w[1][0]

def world_to_pixel(X,Y):
    pt = np.array([X,Y,1]).reshape(3,1)
    img = H_inv@pt; img/=img[2]
    return int(img[0][0]), int(img[1][0])

def smooth_points(points):
    pts=list(points)
    sm=[]
    for i in range(len(pts)):
        xs=[p[0] for p in pts[max(0,i-3):i+1]]
        ys=[p[1] for p in pts[max(0,i-3):i+1]]
        sm.append((int(np.mean(xs)),int(np.mean(ys))))
    return sm

def compute_velocity(hist):
    vel=np.zeros(2); count=0
    for i in range(1,min(6,len(hist))):
        vel+=(hist[-i]-hist[-i-1])/dt
        count+=1
    return vel/max(count,1)

def compute_speed(vel, tid):
    speed=np.linalg.norm(vel)*3.6
    speed=0.7*track_speed[tid]+0.3*speed
    track_speed[tid]=speed
    return speed

def draw_axes(frame):
    for x in np.arange(0,3.1,1):
        p=world_to_pixel(x,0)
        cv2.putText(frame,f"{int(x)}m",p,cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,0,0),1)

    for y in range(0,75,10):
        p=world_to_pixel(0,y)
        cv2.putText(frame,f"{y}m",p,cv2.FONT_HERSHEY_SIMPLEX,0.4,(0,255,0),1)

def draw_traj(frame, pts):
    for i in range(1,len(pts)):
        if i%2==0:
            cv2.line(frame, pts[i-1], pts[i], (255,0,255),2)

def draw_glow(frame, center):
    for r in range(20,0,-4):
        overlay=frame.copy()
        cv2.circle(overlay,center,r,(0,0,255),-1)
        cv2.addWeighted(overlay,0.2,frame,0.8,0,frame)

# ---- MAIN ----
while cap.isOpened():
    ret,frame=cap.read()
    if not ret: break

    frame=cv2.resize(frame,(960,540))

    # ROI
    roi_pts=np.array([
        [540*scale_x,1066*scale_y],
        [1650*scale_x,1066*scale_y],
        [1080*scale_x,216*scale_y],
        [972*scale_x,216*scale_y]
    ],dtype=np.int32)

    mask=np.zeros(frame.shape[:2],dtype=np.uint8)
    cv2.fillPoly(mask,[roi_pts],255)
    frame_masked=cv2.bitwise_and(frame,frame,mask=mask)

    cv2.polylines(frame,[roi_pts],True,(255,0,0),2)
    draw_axes(frame)

    results=model(frame_masked)[0]

    detections=[]
    for b in results.boxes:
        x1,y1,x2,y2=b.xyxy[0].cpu().numpy()
        detections.append([x1,y1,x2,y2,1.0])

    tracks=tracker.update(np.array(detections) if detections else np.empty((0,5)))

    vehicles={}

    for t in tracks:
        x1,y1,x2,y2,tid=t
        tid=int(tid)

        u=(x1+x2)/2; v=y2
        X,Y=pixel_to_world(u,v)

        hist=track_history_world[tid]
        hist.append(np.array([X,Y]))

        vel=compute_velocity(hist)
        speed=compute_speed(vel,tid)

        vehicles[tid]={"pos":np.array([X,Y]),"vel":vel,"px":(int(u),int(v))}

        track_history_px[tid].append((int(u),int(v)))
        pts=smooth_points(track_history_px[tid])
        draw_traj(frame,pts)
        #  RESTORED BOX + LABEL
        cv2.rectangle(frame,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
        cv2.putText(frame,f"ID:{tid} {speed:.1f}km/h",
                    (int(x1),int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,255),2)

    #  TRUE TRAJECTORY INTERSECTION ONLY
    for id1,v1 in vehicles.items():

        best=None
        best_t=1e9

        for id2,v2 in vehicles.items():
            if id1==id2: continue

            p1=v1["pos"]; v1v=v1["vel"]
            p2=v2["pos"]; v2v=v2["vel"]

            rel_p=p2-p1
            rel_v=v2v-v1v

            if np.linalg.norm(rel_v)<1e-3:
                continue

            t=-np.dot(rel_p,rel_v)/np.dot(rel_v,rel_v)

            if t<=0 or t>5:
                continue

            #  STRICT INTERSECTION CONDITION
            future_p1=p1+v1v*t
            future_p2=p2+v2v*t

            if np.linalg.norm(future_p1-future_p2) > 1.0:
                continue

            if t<best_t:
                best_t=t
                best=(id2,v2,t)

        if best is None:
            continue

        id2,v2,t=best
        collision_point=v1["pos"]+v1["vel"]*t
        px=world_to_pixel(collision_point[0],collision_point[1])

        draw_glow(frame,px)
        cv2.line(frame,v1["px"],px,(0,0,255),2)
        cv2.line(frame,v2["px"],px,(0,0,255),2)

    out.write(frame)

cap.release()
out.release()j

print(" FINAL: correct trajectories + single collision + stable")


0: 384x640 2 cars, 2 2wheelers, 25.5ms
Speed: 2.1ms preprocess, 25.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.4ms
Speed: 2.2ms preprocess, 24.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.4ms
Speed: 2.8ms preprocess, 24.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 19.4ms
Speed: 2.1ms preprocess, 19.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 18.1ms
Speed: 2.1ms preprocess, 18.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 18.1ms
Speed: 2.1ms preprocess, 18.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 17.8ms
Speed: 2.0ms preprocess, 17.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 17.8ms
S

In [10]:
print(model.names)

{0: 'car', 1: 'person', 2: '2wheeler'}


In [16]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import math

# ---- LOAD SORT ----
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ---- MODEL ----
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt")

# ---- VIDEO ----
cap = cv2.VideoCapture("/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4")

FPS = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1: FPS = 30
dt = 1/FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("/kaggle/working/final_output.mp4", fourcc, FPS, (960,540))

tracker = Sort(max_age=20, min_hits=3, iou_threshold=0.3)

# ---- STORAGE ----
track_history_world = defaultdict(lambda: deque(maxlen=20))
track_history_px = defaultdict(lambda: deque(maxlen=20))
track_speed = defaultdict(float)
track_class = {}

class_names = {
    0: "Car",
    1: "Person",
    2: "2Wheeler"
}

# ---- HOMOGRAPHY ----
scale_x = 960/1920
scale_y = 540/1080

img_pts = np.array([
    [540*scale_x,1066*scale_y],
    [1650*scale_x,1066*scale_y],
    [972*scale_x,216*scale_y],
    [1080*scale_x,216*scale_y]
], dtype=np.float32)

world_pts = np.array([
    [0,0],[3,0],[0,74.65],[3,74.65]
], dtype=np.float32)

H,_ = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

# ---- FUNCTIONS ----
def pixel_to_world(u,v):
    pt = np.array([u,v,1]).reshape(3,1)
    w = H@pt; w/=w[2]
    return w[0][0], w[1][0]

def world_to_pixel(X,Y):
    pt = np.array([X,Y,1]).reshape(3,1)
    img = H_inv@pt; img/=img[2]
    return int(img[0][0]), int(img[1][0])

def smooth_points(points):
    pts=list(points)
    sm=[]
    for i in range(len(pts)):
        xs=[p[0] for p in pts[max(0,i-3):i+1]]
        ys=[p[1] for p in pts[max(0,i-3):i+1]]
        sm.append((int(np.mean(xs)),int(np.mean(ys))))
    return sm

def compute_velocity(hist):
    vel=np.zeros(2); count=0
    for i in range(1,min(6,len(hist))):
        vel+=(hist[-i]-hist[-i-1])/dt
        count+=1
    return vel/max(count,1)

def compute_speed(vel, tid):
    speed=np.linalg.norm(vel)*3.6
    speed=0.7*track_speed[tid]+0.3*speed
    track_speed[tid]=speed
    return speed

def draw_traj(frame, pts):
    for i in range(1,len(pts)):
        if i%2==0:
            cv2.line(frame, pts[i-1], pts[i], (255,0,255),2)

def draw_glow(frame, center):
    for r in range(20,0,-4):
        overlay=frame.copy()
        cv2.circle(overlay,center,r,(0,0,255),-1)
        cv2.addWeighted(overlay,0.2,frame,0.8,0,frame)

# ---- MAIN ----
while cap.isOpened():
    ret,frame=cap.read()
    if not ret: break

    frame=cv2.resize(frame,(960,540))

    # ROI
    roi_pts=np.array([
        [540*scale_x,1066*scale_y],
        [1650*scale_x,1066*scale_y],
        [1080*scale_x,216*scale_y],
        [972*scale_x,216*scale_y]
    ],dtype=np.int32)

    mask=np.zeros(frame.shape[:2],dtype=np.uint8)
    cv2.fillPoly(mask,[roi_pts],255)
    frame_masked=cv2.bitwise_and(frame,frame,mask=mask)

    cv2.polylines(frame,[roi_pts],True,(255,0,0),2)

    results=model(frame_masked)[0]

    detections=[]
    track_class.clear()

    # ---- DETECTIONS WITH CLASS FILTER ----
    for b in results.boxes:
        cls = int(b.cls[0])
        conf = float(b.conf[0])

        if cls not in [0,1,2]:
            continue
        if conf < 0.4:
            continue

        x1,y1,x2,y2 = b.xyxy[0].cpu().numpy()
        detections.append([x1,y1,x2,y2,conf])

        cx = int((x1+x2)/2)
        cy = int((y1+y2)/2)
        track_class[(cx,cy)] = cls

    tracks=tracker.update(np.array(detections) if detections else np.empty((0,5)))

    vehicles={}

    for t in tracks:
        x1,y1,x2,y2,tid=t
        tid=int(tid)

        cx = int((x1+x2)/2)
        cy = int((y1+y2)/2)
        cls = track_class.get((cx,cy), -1)

        u=(x1+x2)/2; v=y2
        X,Y=pixel_to_world(u,v)

        hist=track_history_world[tid]
        hist.append(np.array([X,Y]))

        if len(hist) < 5:
            continue

        vel=compute_velocity(hist)
        speed=compute_speed(vel,tid)

        vehicles[tid]={"pos":np.array([X,Y]),"vel":vel,"px":(int(u),int(v))}

        # trajectory
        track_history_px[tid].append((int(u),int(v)))
        pts=smooth_points(track_history_px[tid])
        draw_traj(frame,pts)

        # future prediction
        for dt_future in np.linspace(0,3,10):
            future = np.array([X,Y]) + vel*dt_future
            px_future = world_to_pixel(future[0], future[1])
            cv2.circle(frame, px_future, 2, (255,255,0), -1)

        label = class_names.get(cls, "Obj")

        cv2.rectangle(frame,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
        cv2.putText(frame,f"{label} ID:{tid} {speed:.1f}km/h",
                    (int(x1),int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,255),2)

    # ---- COLLISION ----
    for id1,v1 in vehicles.items():

        best=None
        best_t=1e9

        for id2,v2 in vehicles.items():
            if id1==id2: continue

            p1=v1["pos"]; v1v=v1["vel"]
            p2=v2["pos"]; v2v=v2["vel"]

            cos_angle = np.dot(v1v, v2v) / (np.linalg.norm(v1v)*np.linalg.norm(v2v)+1e-6)
            if cos_angle > 0.8:
                continue

            rel_p=p2-p1
            rel_v=v2v-v1v

            if np.linalg.norm(rel_v)<1e-3:
                continue

            t=-np.dot(rel_p,rel_v)/np.dot(rel_v,rel_v)

            if t<=0 or t>5:
                continue

            future_p1=p1+v1v*t
            future_p2=p2+v2v*t

            if np.linalg.norm(future_p1-future_p2) > 0.5:
                continue

            if t<best_t:
                best_t=t
                best=(id2,v2,t)

        if best is None:
            continue

        id2,v2,t=best
        collision_point=v1["pos"]+v1["vel"]*t
        px=world_to_pixel(collision_point[0],collision_point[1])

        # risk color
        if t < 1:
            color=(0,0,255)
        elif t < 2.5:
            color=(0,255,255)
        else:
            color=(0,255,0)

        draw_glow(frame,px)
        cv2.line(frame,v1["px"],px,color,2)
        cv2.line(frame,v2["px"],px,color,2)

        cv2.putText(frame,f"TTC:{t:.2f}s",px,
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

    cv2.putText(frame,f"Vehicles: {len(vehicles)}",
                (20,30),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

    out.write(frame)

cap.release()
out.release()

print("FINAL MULTI-CLASS SMART SYSTEM READY 🚀")


0: 384x640 2 cars, 2 2wheelers, 25.6ms
Speed: 2.5ms preprocess, 25.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.6ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.0ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.3ms preprocess, 24.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 16.5ms
Speed: 2.1ms preprocess, 16.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 16.6ms
Speed: 2.9ms preprocess, 16.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 16.5ms
Speed: 1.9ms preprocess, 16.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 16.6ms
S

FINAL CODE TO SHOW


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import math
import itertools

# ---- LOAD SORT ----
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ---- MODEL ----
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt")

# ---- VIDEO ----
cap = cv2.VideoCapture("/kaggle/input/datasets/shashwatkumarjha/finalchoticlip/ChotiClip.mp4")

FPS = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1: FPS = 30
dt = 1/FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("/kaggle/working/final_yuhfdvdfvoutput.mp4", fourcc, int(FPS), (960,540))

tracker = Sort(max_age=20, min_hits=3, iou_threshold=0.3)

# ---- STORAGE ----
track_history_world = defaultdict(lambda: deque(maxlen=20))
track_history_px = defaultdict(lambda: deque(maxlen=20))
track_speed = defaultdict(float)
velocity_ema = defaultdict(lambda: np.zeros(2))
track_class = {}

ALPHA = 0.1

class_names = {
    0: "Car",
    1: "Person",
    2: "2Wheeler"
}

# ---- HOMOGRAPHY ----
scale_x = 960/1920
scale_y = 540/1080

img_pts = np.array([
    [972*scale_x, 216*scale_y],
    [1080*scale_x, 216*scale_y],
    [1650*scale_x, 1066*scale_y],
    [540*scale_x, 1066*scale_y]
], dtype=np.float32)

world_pts = np.array([
    [0, 74.65],
    [3, 74.65],
    [3, 0],
    [0, 0]
], dtype=np.float32)

H, _ = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

# ---- FUNCTIONS ----
def pixel_to_world(u, v):
    pt = np.array([u, v, 1]).reshape(3,1)
    w = H @ pt
    w /= w[2]
    return w[0][0], w[1][0]

def world_to_pixel(X, Y):
    pt = np.array([X, Y, 1]).reshape(3,1)
    img = H_inv @ pt
    img /= img[2]
    return int(img[0][0]), int(img[1][0])

def smooth_points(points):
    pts = list(points)
    sm = []
    for i in range(len(pts)):
        xs = [p[0] for p in pts[max(0, i-3):i+1]]
        ys = [p[1] for p in pts[max(0, i-3):i+1]]
        sm.append((int(np.mean(xs)), int(np.mean(ys))))
    return sm

def compute_velocity(hist, tid):
    if len(hist) < 2:
        return np.zeros(2)
    current_vel = (hist[-1] - hist[-2]) / dt
    smoothed_vel = (ALPHA * current_vel) + ((1 - ALPHA) * velocity_ema[tid])
    velocity_ema[tid] = smoothed_vel
    return smoothed_vel

def compute_speed(vel, tid):
    speed = np.linalg.norm(vel) * 3.6
    speed = 0.7 * track_speed[tid] + 0.3 * speed
    track_speed[tid] = speed
    return speed

def draw_axes(frame):
    for x in np.arange(0, 3.1, 1):
        p = world_to_pixel(x, 0)
        cv2.putText(frame, f"{int(x)}m", p, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,0,0), 1)

    for y in range(0, 75, 10):
        p = world_to_pixel(0, y)
        cv2.putText(frame, f"{y}m", p, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)

def draw_traj(frame, pts):
    for i in range(1, len(pts)):
        if i % 2 == 0:
            cv2.line(frame, pts[i-1], pts[i], (255,0,255), 2)

def draw_glow(frame, center):
    for r in range(20, 0, -4):
        overlay = frame.copy()
        cv2.circle(overlay, center, r, (0,0,255), -1)
        cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)

# ---- MAIN ----
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.resize(frame, (960, 540))

    roi_pts = np.array([
        [972*scale_x, 216*scale_y],
        [1080*scale_x, 216*scale_y],
        [1650*scale_x, 1066*scale_y],
        [540*scale_x, 1066*scale_y]
    ], dtype=np.int32)

    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [roi_pts], 255)
    frame_masked = cv2.bitwise_and(frame, frame, mask=mask)

    cv2.polylines(frame, [roi_pts], True, (255,0,0), 2)
    draw_axes(frame)

    results = model(frame_masked)[0]

    detections = []
    track_class.clear()

    # ✅ FIXED CLASS HANDLING
    for b in results.boxes:
        cls = int(b.cls[0].cpu().numpy())
        conf = float(b.conf[0].cpu().numpy())

        if cls not in [0,1,2]:
            continue
        if conf < 0.4:
            continue

        x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
        detections.append([x1, y1, x2, y2, conf])

        cx = int((x1 + x2)/2)
        cy = int((y1 + y2)/2)
        track_class[(cx, cy)] = cls

    tracks = tracker.update(np.array(detections) if detections else np.empty((0,5)))

    vehicles = {}

    for t in tracks:
        x1, y1, x2, y2, tid = t
        tid = int(tid)

        cx = int((x1 + x2)/2)
        cy = int((y1 + y2)/2)
        cls = track_class.get((cx, cy), -1)

        u = (x1 + x2) / 2
        v = y2
        X, Y = pixel_to_world(u, v)

        hist = track_history_world[tid]
        hist.append(np.array([X, Y]))

        vel = compute_velocity(hist, tid)
        speed = compute_speed(vel, tid)

        vehicles[tid] = {"pos": np.array([X, Y]), "vel": vel, "px": (int(u), int(v))}

        track_history_px[tid].append((int(u), int(v)))
        pts = smooth_points(track_history_px[tid])
        draw_traj(frame, pts)

        #  FUTURE TRAJECTORY
        future_pts_px = []
        if np.linalg.norm(vel) > 0.5:
            for step in np.arange(0, 5.5, 0.5):
                fut_X = X + vel[0] * step
                fut_Y = Y + vel[1] * step
                px_x, px_y = world_to_pixel(fut_X, fut_Y)
                future_pts_px.append((px_x, px_y))

            for i in range(1, len(future_pts_px)):
                if i % 2 != 0:
                    cv2.line(frame, future_pts_px[i-1], future_pts_px[i], (0,165,255), 2)

        label = class_names.get(cls, "Obj")

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
        cv2.putText(frame, f"{label} ID:{tid} {speed:.1f}km/h",
                    (int(x1), int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)

    #  COLLISION
    for (id1, v1), (id2, v2) in itertools.combinations(vehicles.items(), 2):

        p1 = v1["pos"]; v1v = v1["vel"]
        p2 = v2["pos"]; v2v = v2["vel"]

        rel_p = p2 - p1
        rel_v = v2v - v1v

        if np.linalg.norm(rel_v) < 1e-3:
            continue

        t = -np.dot(rel_p, rel_v) / np.dot(rel_v, rel_v)

        if t <= 0 or t > 5:
            continue

        future_p1 = p1 + v1v * t
        future_p2 = p2 + v2v * t

        if abs(future_p1[0] - future_p2[0]) > 1.0:
            continue
        if abs(future_p1[1] - future_p2[1]) > 4.0:
            continue

        collision_point = (future_p1 + future_p2) / 2
        px = world_to_pixel(collision_point[0], collision_point[1])

        draw_glow(frame, px)
        cv2.line(frame, v1["px"], px, (0,0,255), 2)
        cv2.line(frame, v2["px"], px, (0,0,255), 2)

    cv2.putText(frame, f"Vehicles: {len(vehicles)}",
                (20,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    out.write(frame)

cap.release()
out.release()

print(" FINAL FIXED MULTI-CLASS + EMA + COLLISION SYSTEM 🚀")


0: 384x640 2 cars, 2 2wheelers, 26.3ms
Speed: 2.2ms preprocess, 26.3ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 24.9ms
Speed: 2.8ms preprocess, 24.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 21.2ms
Speed: 2.1ms preprocess, 21.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 21.2ms
Speed: 2.1ms preprocess, 21.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 21.2ms
Speed: 2.0ms preprocess, 21.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 21.2ms
Speed: 2.2ms preprocess, 21.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 20.4ms
Speed: 2.6ms preprocess, 20.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 2 2wheelers, 18.3ms
S